# NYC Yellow Taxi Trip Data Analysis with PySpark

Analysis of 3.5M NYC yellow taxi trips (August 2025) using PySpark.  
Covers data cleaning, exploratory analysis, SQL queries, and performance optimization.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('NYC Yellow Taxi Trip Data').getOrCreate()

## Environment Setup
Initializing Spark session with NYC Yellow Taxi Trip Data app configuration.

In [ ]:
print(f"Spark version: {spark.version}")
print(f"Application name: {spark.sparkContext.appName}")
print(f"Master: {spark.sparkContext.master}")
print(f"Python version: {spark.sparkContext.pythonVer}")
print(f"Default parallelism: {spark.sparkContext.defaultParallelism}")

Spark version: 4.0.1
Application name: NYC Yellow Taxi Trip Data
Master: local[*]
Python version: 3.12
Default parallelism: 2


## Data Loading
Loading raw parquet file containing trip records for August 2025.

In [ ]:
df = spark.read.parquet('/content/yellow_tripdata_2025-08.parquet')

In [ ]:
df = df.withColumnRenamed('tpep_pickup_datetime', 'pickup_datetime').withColumnRenamed('tpep_dropoff_datetime', 'dropoff_datetime')

In [ ]:
print(f"Initial record count: {df.count():,}")
print(f"Number of columns: {len(df.columns)}")

Initial record count: 3,574,091
Number of columns: 20


In [ ]:
df.dtypes

[('VendorID', 'int'),
 ('pickup_datetime', 'timestamp_ntz'),
 ('dropoff_datetime', 'timestamp_ntz'),
 ('passenger_count', 'bigint'),
 ('trip_distance', 'double'),
 ('RatecodeID', 'bigint'),
 ('store_and_fwd_flag', 'string'),
 ('PULocationID', 'int'),
 ('DOLocationID', 'int'),
 ('payment_type', 'bigint'),
 ('fare_amount', 'double'),
 ('extra', 'double'),
 ('mta_tax', 'double'),
 ('tip_amount', 'double'),
 ('tolls_amount', 'double'),
 ('improvement_surcharge', 'double'),
 ('total_amount', 'double'),
 ('congestion_surcharge', 'double'),
 ('Airport_fee', 'double'),
 ('cbd_congestion_fee', 'double')]

## Data Cleaning

### Null Values
Checking for missing values across all columns.

In [ ]:
from pyspark.sql.functions import col, when, count, isnan, trim

df.select([
    count(when(col(c).isNull() | (trim(col(c)) == '') | isnan(col(c)), c)).alias(c)
    if dict(df.dtypes)[c] in ['double', 'float']
    else count(when(col(c).isNull() | (trim(col(c)) == ''), c)).alias(c)
    for c in df.columns
]).show()


+--------+---------------+----------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|pickup_datetime|dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+---------------+----------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       0|              0|               0|         886234|            0|    886234|            886234|           0|           0|           0|          0|    0|

### Filtering
Removing invalid records: negative durations, zero fares, zero distances, and unrealistic passenger counts.

In [ ]:
df_clean = df.dropna()
print(f"Number of records after dropping nulls: {df_clean.count():,}")

Number of records after dropping nulls: 2,687,857


In [ ]:
from pyspark.sql.types import IntegerType

df_clean = df_clean \
    .withColumn('passenger_count', col('passenger_count').cast(IntegerType())) \
    .withColumn('RatecodeID', col('RatecodeID').cast(IntegerType())) \
    .withColumn('payment_type', col('payment_type').cast(IntegerType()))

In [ ]:
df_clean = df_clean.filter(
    (col('pickup_datetime') < col('dropoff_datetime')) &
    (col('total_amount') > 0) &
    (col('trip_distance') > 0) &
    (col('fare_amount') > 0) & (col('fare_amount') < 500) &
    (col('passenger_count') > 0) & (col('passenger_count') <= 6)
)

In [ ]:
print(f"Number of records after filtering invalid entries: {df_clean.count():,}")

Number of records after filtering invalid entries: 2,511,464


In [ ]:
df_clean.select([
    count(when(col(c).isNull() | (trim(col(c)) == '') | isnan(col(c)), c)).alias(c)
    if dict(df_clean.dtypes)[c] in ['double', 'float']
    else count(when(col(c).isNull() | (trim(col(c)) == ''), c)).alias(c)
    for c in df_clean.columns
]).show()

+--------+---------------+----------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|pickup_datetime|dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+---------------+----------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       0|              0|               0|              0|            0|         0|                 0|           0|           0|           0|          0|    0|

In [ ]:
df_clean.describe().show()

+-------+-------------------+------------------+------------------+------------------+------------------+-----------------+------------------+------------------+------------------+------------------+-------------------+------------------+------------------+---------------------+------------------+--------------------+-------------------+------------------+
|summary|           VendorID|   passenger_count|     trip_distance|        RatecodeID|store_and_fwd_flag|     PULocationID|      DOLocationID|      payment_type|       fare_amount|             extra|            mta_tax|        tip_amount|      tolls_amount|improvement_surcharge|      total_amount|congestion_surcharge|        Airport_fee|cbd_congestion_fee|
+-------+-------------------+------------------+------------------+------------------+------------------+-----------------+------------------+------------------+------------------+------------------+-------------------+------------------+------------------+---------------------+---

In [ ]:
df_clean.select(
    'trip_distance',
    'fare_amount',
    'extra',
    'mta_tax',
    'tip_amount',
    'tolls_amount',
    'total_amount',
    'passenger_count'
).describe().show()

+-------+------------------+------------------+------------------+-------------------+------------------+------------------+------------------+------------------+
|summary|     trip_distance|       fare_amount|             extra|            mta_tax|        tip_amount|      tolls_amount|      total_amount|   passenger_count|
+-------+------------------+------------------+------------------+-------------------+------------------+------------------+------------------+------------------+
|  count|           2511464|           2511464|           2511464|            2511464|           2511464|           2511464|           2511464|           2511464|
|   mean|3.7276851230993837|20.495627729485598|1.5676710078265106|0.49207285073566653|3.6371542773455796|0.6394647982196007| 30.21432789799862| 1.334373496892649|
| stddev| 5.426771629285761|19.564801192414688| 1.921356897693482|0.06259180102008156| 4.203460991076329|2.3278150762917353|24.152134268712494|0.7533217088422028|
|    min|             

In [ ]:
df_clean.dtypes

[('VendorID', 'int'),
 ('pickup_datetime', 'timestamp_ntz'),
 ('dropoff_datetime', 'timestamp_ntz'),
 ('passenger_count', 'int'),
 ('trip_distance', 'double'),
 ('RatecodeID', 'int'),
 ('store_and_fwd_flag', 'string'),
 ('PULocationID', 'int'),
 ('DOLocationID', 'int'),
 ('payment_type', 'int'),
 ('fare_amount', 'double'),
 ('extra', 'double'),
 ('mta_tax', 'double'),
 ('tip_amount', 'double'),
 ('tolls_amount', 'double'),
 ('improvement_surcharge', 'double'),
 ('total_amount', 'double'),
 ('congestion_surcharge', 'double'),
 ('Airport_fee', 'double'),
 ('cbd_congestion_fee', 'double')]

In [ ]:
df_clean.show(10, truncate=False)

+--------+-------------------+-------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|pickup_datetime    |dropoff_datetime   |passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+-------------------+-------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|2       |2025-08-01 00:52:23|2025-08-01 01:12:20|1              |8.44         |1         |N                 |138         |141         |1  

### Summary
Overview of records removed and final dataset shape.

In [ ]:
print(f"Original records: {df.count():,}")
print(f"Clean records: {df_clean.count():,}")
print(f"Records removed: {df.count() - df_clean.count():,}")
print(f"Total columns: {len(df_clean.columns)}")

Original records: 3,574,091
Clean records: 2,511,464
Records removed: 1,062,627
Total columns: 20


In [ ]:
df_clean.write.mode('overwrite').parquet('/content/cleaned_yellow_tripdata_2025-08.parquet')

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark = SparkSession.builder.appName('NYC Taxi Analytics').config('spark.sql.adaptive.enabled', 'true').getOrCreate()

print(f"Spark version: {spark.version}")

Spark version: 4.0.1


In [ ]:
df = spark.read.parquet('/content/cleaned_yellow_tripdata_2025-08.parquet')

print(f"Records loaded: {df.count():,}")
print(f"Columns: {len(df.columns)}")

df.show(5, truncate=False)

Records loaded: 2,511,464
Columns: 20
+--------+-------------------+-------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|pickup_datetime    |dropoff_datetime   |passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+-------------------+-------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|2       |2025-08-01 00:52:23|2025-08-01 01:12:20|1              |8.44         |1         |N         

## Exploratory Data Analysis

### Feature Engineering
Adding derived columns: trip duration, pickup hour/day, fare per mile, tip percentage, and day name.

In [ ]:
df_enhanced = df \
    .withColumn('trip_duration_minutes',
                round((unix_timestamp(col('dropoff_datetime')) -
                       unix_timestamp(col('pickup_datetime'))) / 60, 2)) \
    .withColumn('pickup_hour', hour(col('pickup_datetime'))) \
    .withColumn('pickup_day', dayofweek(col('pickup_datetime'))) \
    .withColumn('fare_per_mile',
                round(col('fare_amount') / col('trip_distance'), 2)) \
    .withColumn('tip_percentage',
                round((col('tip_amount') / col('fare_amount')) * 100, 2)) \
    .withColumn('day_name',
                when(col('pickup_day') == 1, 'Sunday')
                .when(col('pickup_day') == 2, 'Monday')
                .when(col('pickup_day') == 3, 'Tuesday')
                .when(col('pickup_day') == 4, 'Wednesday')
                .when(col('pickup_day') == 5, 'Thursday')
                .when(col('pickup_day') == 6, 'Friday')
                .when(col('pickup_day') == 7, 'Saturday'))

df_enhanced.select('pickup_datetime', 'dropoff_datetime', 'trip_duration_minutes',
                   'pickup_hour', 'day_name', 'fare_per_mile', 'tip_percentage').show(10)

+-------------------+-------------------+---------------------+-----------+--------+-------------+--------------+
|    pickup_datetime|   dropoff_datetime|trip_duration_minutes|pickup_hour|day_name|fare_per_mile|tip_percentage|
+-------------------+-------------------+---------------------+-----------+--------+-------------+--------------+
|2025-08-01 00:52:23|2025-08-01 01:12:20|                19.95|          0|  Friday|          4.0|         14.79|
|2025-08-01 00:03:01|2025-08-01 00:15:33|                12.53|          0|  Friday|         4.26|           0.0|
|2025-08-01 00:25:34|2025-08-01 00:33:18|                 7.73|          0|  Friday|         5.33|         22.54|
|2025-08-01 00:16:36|2025-08-01 00:33:41|                17.08|          0|  Friday|         6.01|           0.0|
|2025-08-01 00:56:02|2025-08-01 01:15:37|                19.58|          0|  Friday|          4.7|          4.05|
|2025-08-01 00:12:10|2025-08-01 00:15:58|                  3.8|          0|  Friday|    

### Long Trips
Filtering trips over 10 miles and $30 fare.

In [ ]:
long_trips = df_enhanced.filter(
    (col('trip_distance') > 10) &
    (col('fare_amount') > 30)
).orderBy(col('trip_distance').desc())

print(f"Long trips (>10 miles, >$30): {long_trips.count():,}")
long_trips.select('trip_distance', 'fare_amount', 'trip_duration_minutes',
                  'PULocationID', 'DOLocationID').show(10)

Long trips (>10 miles, >$30): 251,429
+-------------+-----------+---------------------+------------+------------+
|trip_distance|fare_amount|trip_duration_minutes|PULocationID|DOLocationID|
+-------------+-----------+---------------------+------------+------------+
|        302.4|       33.5|                28.83|         200|          42|
|       244.57|      300.0|               262.68|         265|         265|
|       197.54|      450.0|               188.35|          93|         265|
|        156.1|      350.0|               216.88|         132|         265|
|       137.78|      230.0|               187.62|          39|         265|
|       129.36|      250.0|               142.68|         233|         265|
|       127.15|      350.0|               205.62|         138|         265|
|       126.34|      350.0|                128.2|         164|         265|
|        126.0|      250.0|               184.25|         132|         265|
|        125.5|      85.25|               233.43| 

### Peak Hours
Analyzing trips during morning and evening rush hours (8–9 AM, 5–6 PM).

In [ ]:
peak_trips = df_enhanced.filter(
    col('pickup_hour').isin([8, 9, 17, 18])
)

print(f"Peak hour trips: {peak_trips.count():,}")
print(f"Percentage of total: {(peak_trips.count() / df_enhanced.count() * 100):.2f}%")

Peak hour trips: 548,917
Percentage of total: 21.86%


### Hourly & Daily Patterns
Aggregating trip count, average fare, distance, duration, and revenue by hour and day of week.

In [ ]:
hourly_stats = df_enhanced.groupBy('pickup_hour').agg(
    count('*').alias('trip_count'),
    round(avg('fare_amount'), 2).alias('avg_fare'),
    round(avg('trip_distance'), 2).alias('avg_distance'),
    round(avg('trip_duration_minutes'), 2).alias('avg_duration'),
    round(sum('total_amount'), 2).alias('total_revenue')
).orderBy('pickup_hour')

hourly_stats.show(24)

+-----------+----------+--------+------------+------------+-------------+
|pickup_hour|trip_count|avg_fare|avg_distance|avg_duration|total_revenue|
+-----------+----------+--------+------------+------------+-------------+
|          0|     68519|   22.07|        4.49|       15.19|   2213334.94|
|          1|     45550|   20.11|        4.02|       13.98|   1350952.76|
|          2|     29181|   17.89|        3.45|       12.71|    781758.55|
|          3|     18789|   18.28|        3.52|       12.96|    512020.69|
|          4|     12487|   23.87|        4.94|        15.8|    420056.97|
|          5|     15478|   29.06|        6.58|       19.13|    605985.69|
|          6|     33838|   23.85|        5.21|        17.6|   1096386.38|
|          7|     60276|    21.0|        4.26|       17.26|   1779745.97|
|          8|     86902|   19.33|        3.56|       16.53|   2416356.51|
|          9|    106081|   19.13|        3.34|       16.91|   2931235.18|
|         10|    118531|    19.4|     

In [ ]:
daily_stats = df_enhanced.groupBy('pickup_day', 'day_name').agg(
    count('*').alias('trip_count'),
    round(avg('fare_amount'), 2).alias('avg_fare'),
    round(avg('trip_distance'), 2).alias('avg_distance'),
    round(sum('total_amount'), 2).alias('total_revenue')
).orderBy('pickup_day')

daily_stats.show()

+----------+---------+----------+--------+------------+-------------+
|pickup_day| day_name|trip_count|avg_fare|avg_distance|total_revenue|
+----------+---------+----------+--------+------------+-------------+
|         1|   Sunday|    353013|   21.74|        4.18|1.105949322E7|
|         2|   Monday|    295922|   21.16|        4.01|    9235219.7|
|         3|  Tuesday|    338415|   19.79|        3.53|1.000760259E7|
|         4|Wednesday|    371998|   19.65|        3.38|1.095804063E7|
|         5| Thursday|    351619|   20.65|        3.66| 1.07679829E7|
|         6|   Friday|    407527|   20.72|        3.77|1.249038068E7|
|         7| Saturday|    392970|   19.92|        3.63|1.136347708E7|
+----------+---------+----------+--------+------------+-------------+



### Location Analysis
Top 10 pickup and dropoff locations by volume.

In [ ]:
top_pickup = df_enhanced.groupBy('PULocationID').agg(
    count('*').alias('pickup_count'),
    round(avg('fare_amount'), 2).alias('avg_fare'),
    round(avg('trip_distance'), 2).alias('avg_distance')
).orderBy(col('pickup_count').desc()).limit(10)

print("Top 10 pickup locations:")
top_pickup.show()

Top 10 pickup locations:
+------------+------------+--------+------------+
|PULocationID|pickup_count|avg_fare|avg_distance|
+------------+------------+--------+------------+
|         132|      169442|   62.06|       15.07|
|         161|      120420|   15.92|        2.31|
|         237|      109051|   12.68|        1.81|
|         186|      103192|   16.72|         2.4|
|         138|       93466|   44.24|        9.52|
|         162|       92670|    15.2|        2.27|
|         230|       86566|   18.92|        3.05|
|         236|       84462|   13.28|        2.04|
|         170|       77805|   15.22|        2.27|
|         163|       69195|   16.07|        2.42|
+------------+------------+--------+------------+



In [ ]:
top_dropoff = df_enhanced.groupBy('DOLocationID').agg(
    count('*').alias('dropoff_count'),
    round(avg('fare_amount'), 2).alias('avg_fare'),
    round(avg('trip_distance'), 2).alias('avg_distance')
).orderBy(col('dropoff_count').desc()).limit(10)

print("Top 10 dropoff locations:")
top_dropoff.show()

Top 10 dropoff locations:
+------------+-------------+--------+------------+
|DOLocationID|dropoff_count|avg_fare|avg_distance|
+------------+-------------+--------+------------+
|         237|        96874|   12.87|        1.89|
|         161|        96783|   15.49|        2.32|
|         236|        91945|   14.04|        2.31|
|         230|        83732|   21.21|        3.53|
|         170|        76937|   15.24|        2.32|
|         162|        73480|   15.25|        2.37|
|          68|        65516|   16.22|        2.63|
|         163|        63744|   16.99|         2.7|
|         239|        63574|   16.17|        2.85|
|         234|        62889|   14.19|        2.13|
+------------+-------------+--------+------------+



### Route Analysis
Most popular origin-destination pairs.

In [ ]:
popular_routes = df_enhanced.groupBy('PULocationID', 'DOLocationID').agg(
    count('*').alias('route_count'),
    round(avg('fare_amount'), 2).alias('avg_fare'),
    round(avg('trip_distance'), 2).alias('avg_distance')
).orderBy(col('route_count').desc()).limit(10)

print("Top 10 popular routes:")
popular_routes.show()

Top 10 popular routes:
+------------+------------+-----------+--------+------------+
|PULocationID|DOLocationID|route_count|avg_fare|avg_distance|
+------------+------------+-----------+--------+------------+
|         237|         236|      13648|    8.18|        1.04|
|         236|         237|      10848|    8.67|        1.02|
|         237|         237|      10120|    7.06|        0.63|
|         132|         265|       9486|  103.88|       20.39|
|         161|         237|       8211|     9.5|        1.06|
|         236|         236|       8014|    6.48|        0.61|
|         237|         161|       7612|    9.75|        1.05|
|         132|         230|       6500|   71.01|       17.76|
|         161|         236|       6260|    13.0|        1.89|
|         186|         230|       6251|   12.63|        1.06|
+------------+------------+-----------+--------+------------+



### Payment Type Breakdown
Comparing trip count, fares, and tipping behavior across payment methods.

In [ ]:
payment_stats = df_enhanced.groupBy('payment_type').agg(
    count('*').alias('trip_count'),
    round(avg('fare_amount'), 2).alias('avg_fare'),
    round(avg('tip_amount'), 2).alias('avg_tip'),
    round(avg('tip_percentage'), 2).alias('avg_tip_pct'),
    round(sum('total_amount'), 2).alias('total_revenue')
).orderBy('payment_type')

print("Statistics by payment type:")
print("1 = Credit Card, 2 = Cash, 3 = No Charge, 4 = Dispute")
payment_stats.show()

Statistics by payment type:
1 = Credit Card, 2 = Cash, 3 = No Charge, 4 = Dispute
+------------+----------+--------+-------+-----------+-------------+
|payment_type|trip_count|avg_fare|avg_tip|avg_tip_pct|total_revenue|
+------------+----------+--------+-------+-----------+-------------+
|           1|   2109833|   20.42|   4.33|      25.04|6.512734485E7|
|           2|    335793|   20.27|    0.0|        0.0|   8753294.01|
|           3|     13873|    21.7|   0.01|       0.11|    384439.42|
|           4|     51965|    24.8|   0.01|       0.05|   1617118.52|
+------------+----------+--------+-------+-----------+-------------+



### Fare Distribution
Grouping trips into fare ranges and examining average distances per range.

In [ ]:
fare_ranges = df_enhanced.withColumn('fare_range',
    when(col('fare_amount') < 10, '0-10')
    .when(col('fare_amount') < 20, '10-20')
    .when(col('fare_amount') < 30, '20-30')
    .when(col('fare_amount') < 50, '30-50')
    .otherwise('50+')
).groupBy('fare_range').agg(
    count('*').alias('trip_count'),
    round(avg('trip_distance'), 2).alias('avg_distance')
).orderBy('fare_range')

print("Fare Range Distribution:")
fare_ranges.show()

Fare Range Distribution:
+----------+----------+------------+
|fare_range|trip_count|avg_distance|
+----------+----------+------------+
|      0-10|    702592|        0.85|
|     10-20|   1065595|        1.94|
|     20-30|    295530|        4.13|
|     30-50|    238390|        8.66|
|       50+|    209357|       16.31|
+----------+----------+------------+



### Passenger Count Analysis
Trip statistics broken down by number of passengers.

In [ ]:
passenger_stats = df_enhanced.groupBy('passenger_count').agg(
    count('*').alias('trip_count'),
    round(avg('fare_amount'), 2).alias('avg_fare'),
    round(avg('trip_distance'), 2).alias('avg_distance'),
    round(avg('tip_amount'), 2).alias('avg_tip')
).orderBy('passenger_count')

print("Statistics by passenger count:")
passenger_stats.show()

Statistics by passenger count:
+---------------+----------+--------+------------+-------+
|passenger_count|trip_count|avg_fare|avg_distance|avg_tip|
+---------------+----------+--------+------------+-------+
|              1|   1966123|   19.55|        3.54|   3.51|
|              2|    361077|   23.27|        4.34|   4.13|
|              3|     96592|   24.29|        4.45|    4.0|
|              4|     71238|    28.0|        4.93|   4.09|
|              5|     10378|   17.64|        3.08|   3.36|
|              6|      6056|   18.98|        3.45|   3.52|
+---------------+----------+--------+------------+-------+



In [ ]:
df_enhanced.createOrReplaceTempView('taxi_trips')

## SQL Queries
Running the same analyses using Spark SQL via a temporary view.

In [ ]:
query1 = spark.sql("""
    SELECT
        pickup_hour,
        COUNT(*) as trip_count,
        ROUND(AVG(fare_amount), 2) as avg_fare,
        ROUND(AVG(trip_distance), 2) as avg_distance,
        ROUND(AVG(trip_duration_minutes), 2) as avg_duration,
        ROUND(SUM(total_amount), 2) as total_revenue
    FROM taxi_trips
    GROUP BY pickup_hour
    ORDER BY pickup_hour
""")

print("Average metrics by hour:")
query1.show(24)

Average metrics by hour:
+-----------+----------+--------+------------+------------+-------------+
|pickup_hour|trip_count|avg_fare|avg_distance|avg_duration|total_revenue|
+-----------+----------+--------+------------+------------+-------------+
|          0|     68519|   22.07|        4.49|       15.19|   2213334.94|
|          1|     45550|   20.11|        4.02|       13.98|   1350952.76|
|          2|     29181|   17.89|        3.45|       12.71|    781758.55|
|          3|     18789|   18.28|        3.52|       12.96|    512020.69|
|          4|     12487|   23.87|        4.94|        15.8|    420056.97|
|          5|     15478|   29.06|        6.58|       19.13|    605985.69|
|          6|     33838|   23.85|        5.21|        17.6|   1096386.38|
|          7|     60276|    21.0|        4.26|       17.26|   1779745.97|
|          8|     86902|   19.33|        3.56|       16.53|   2416356.51|
|          9|    106081|   19.13|        3.34|       16.91|   2931235.18|
|         10|

In [ ]:
query2 = spark.sql("""
    SELECT
        CASE
            WHEN pickup_hour IN (7, 8, 9, 17, 18, 19) THEN 'Peak Hours'
            ELSE 'Off-Peak Hours'
        END as time_period,
        COUNT(*) as trip_count,
        ROUND(AVG(fare_amount), 2) as avg_fare,
        ROUND(AVG(trip_distance), 2) as avg_distance,
        ROUND(AVG(trip_duration_minutes), 2) as avg_duration
    FROM taxi_trips
    GROUP BY time_period
    ORDER BY time_period DESC
""")

print("Peak vs off-peak analysis:")
query2.show()

Peak vs off-peak analysis:
+--------------+----------+--------+------------+------------+
|   time_period|trip_count|avg_fare|avg_distance|avg_duration|
+--------------+----------+--------+------------+------------+
|    Peak Hours|    764928|   19.69|        3.53|       17.14|
|Off-Peak Hours|   1746536|   20.85|        3.81|       17.49|
+--------------+----------+--------+------------+------------+



In [ ]:
query3 = spark.sql("""
    SELECT
        CASE
            WHEN pickup_day IN (1, 7) THEN 'Weekend'
            ELSE 'Weekday'
        END as day_type,
        COUNT(*) as trip_count,
        ROUND(AVG(fare_amount), 2) as avg_fare,
        ROUND(AVG(trip_distance), 2) as avg_distance,
        ROUND(AVG(tip_percentage), 2) as avg_tip_pct
    FROM taxi_trips
    GROUP BY day_type
    ORDER BY day_type
""")

print("Weekend vs weekday analysis:")
query3.show()

Weekend vs weekday analysis:
+--------+----------+--------+------------+-----------+
|day_type|trip_count|avg_fare|avg_distance|avg_tip_pct|
+--------+----------+--------+------------+-----------+
| Weekday|   1765481|   20.38|        3.66|       21.1|
| Weekend|    745983|   20.78|        3.89|      20.89|
+--------+----------+--------+------------+-----------+



In [ ]:
query4 = spark.sql("""
    SELECT
        PULocationID,
        DOLocationID,
        COUNT(*) as trip_count,
        ROUND(AVG(fare_amount), 2) as avg_fare,
        ROUND(AVG(trip_distance), 2) as avg_distance,
        ROUND(SUM(total_amount), 2) as total_revenue
    FROM taxi_trips
    WHERE fare_amount > 50
    GROUP BY PULocationID, DOLocationID
    HAVING COUNT(*) > 10
    ORDER BY total_revenue DESC
    LIMIT 10
""")

print("High-value routes (Fare > $50):")
query4.show()

High-value routes (Fare > $50):
+------------+------------+----------+--------+------------+-------------+
|PULocationID|DOLocationID|trip_count|avg_fare|avg_distance|total_revenue|
+------------+------------+----------+--------+------------+-------------+
|         132|         265|      6251|  138.03|       26.57|   1021716.97|
|         132|         230|      6490|   71.05|       17.77|    618848.87|
|         132|         164|      3386|   70.91|       17.15|    327748.15|
|         132|          48|      3135|   70.98|       17.97|    298111.89|
|         138|         265|      1798|  112.85|       22.04|    256906.03|
|         132|         163|      2706|   70.96|       17.87|    254965.74|
|         132|         161|      2404|   71.05|       17.52|     232780.9|
|         132|          68|      2325|   70.91|       17.95|    225034.68|
|         132|         170|      2322|   70.83|       16.76|    224360.44|
|         230|         132|      2330|   71.99|       17.29|    2178

In [ ]:
query5 = spark.sql("""
    SELECT
        pickup_hour,
        payment_type,
        COUNT(*) as trip_count,
        ROUND(AVG(tip_amount), 2) as avg_tip,
        ROUND(AVG(tip_percentage), 2) as avg_tip_pct
    FROM taxi_trips
    WHERE payment_type = 1  -- Credit card only
    GROUP BY pickup_hour, payment_type
    ORDER BY pickup_hour
""")

print("Tipping patterns by hour (Credit card):")
query5.show(24)

Tipping patterns by hour (Credit card):
+-----------+------------+----------+-------+-----------+
|pickup_hour|payment_type|trip_count|avg_tip|avg_tip_pct|
+-----------+------------+----------+-------+-----------+
|          0|           1|     58269|   4.53|      24.32|
|          1|           1|     38813|   4.05|      24.08|
|          2|           1|     24815|    3.6|      34.02|
|          3|           1|     15829|   3.61|      23.99|
|          4|           1|     10011|   4.39|       22.9|
|          5|           1|     11982|   4.86|      21.12|
|          6|           1|     27650|   4.01|      21.47|
|          7|           1|     51290|   3.86|      24.71|
|          8|           1|     75039|   3.81|      23.41|
|          9|           1|     89806|   3.88|      23.54|
|         10|           1|     97629|   3.96|      27.28|
|         11|           1|    105060|   4.09|      23.93|
|         12|           1|    115375|   4.24|      23.82|
|         13|           1|    12

## Performance Optimization

### Caching
Comparing execution time with and without DataFrame caching.

In [ ]:
import time

start_time = time.time()

result1 = df_enhanced.groupBy('pickup_hour').count().collect()
result2 = df_enhanced.groupBy('pickup_day').count().collect()
result3 = df_enhanced.groupBy('payment_type').count().collect()

time_without_cache = time.time() - start_time
print(f"Execution time without caching: {time_without_cache:.2f} seconds")

Execution time without caching: 1.25 seconds


In [ ]:
df_enhanced.cache()

df_enhanced.count()

start_time = time.time()

result1 = df_enhanced.groupBy('pickup_hour').count().collect()
result2 = df_enhanced.groupBy('pickup_day').count().collect()
result3 = df_enhanced.groupBy('payment_type').count().collect()

time_with_cache = time.time() - start_time
print(f"Execution time with caching: {time_with_cache:.2f} seconds")

speedup = (time_without_cache - time_with_cache) / time_without_cache * 100
print(f"Performance improvement: {speedup:.1f}%")

Execution time with caching: 0.94 seconds
Performance improvement: 24.9%


### Broadcast Join
Joining with the NYC taxi zone lookup table using broadcast hint for small-table optimization.

In [ ]:
df_partitioned = df_enhanced.repartition('pickup_hour')

print(f"Number of partitions: {df_partitioned.rdd.getNumPartitions()}")

start_time = time.time()
hourly_analysis = df_partitioned.filter(col('pickup_hour') == 8).count()
partition_time = time.time() - start_time

print(f"Query on partitioned data: {partition_time:.2f} seconds")
print(f"Records for hour 8: {hourly_analysis:,}")

Number of partitions: 4
Query on partitioned data: 0.45 seconds
Records for hour 8: 86,902


### Partitioning
Repartitioning by pickup hour for faster hour-based filtering.

In [ ]:
location_df = spark.read.csv('taxi_zone_lookup.csv', header=True, inferSchema=True)

print("NYC Taxi Zone Lookup Table:")
location_df.show(10)

location_df = location_df.withColumnRenamed('LocationID', 'LocationID') \
                         .withColumnRenamed('Borough', 'Borough') \
                         .withColumnRenamed('Zone', 'LocationName')

print(f"Total taxi zones: {location_df.count()}")
location_df.show(10)

NYC Taxi Zone Lookup Table:
+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 10 rows
Total taxi zones: 265
+----------+-------------+--------------------+------------+
|LocationI

In [ ]:
start_time = time.time()
regular_join = df_enhanced.join(location_df,
                                 df_enhanced.PULocationID == location_df.LocationID)
regular_count = regular_join.count()
regular_time = time.time() - start_time

print(f"Regular join time: {regular_time:.2f} seconds")
print(f"Joined records: {regular_count:,}")

Regular join time: 0.93 seconds
Joined records: 2,511,464


In [ ]:
start_time = time.time()
broadcast_join = df_enhanced.join(
    broadcast(location_df),
    df_enhanced.PULocationID == location_df.LocationID
)
broadcast_count = broadcast_join.count()
broadcast_time = time.time() - start_time

print(f"Broadcast join time: {broadcast_time:.2f} seconds")
print(f"Joined records: {broadcast_count:,}")

broadcast_join.select('pickup_datetime', 'LocationName', 'Borough',
                      'fare_amount', 'trip_distance').show(10)

Broadcast join time: 0.95 seconds
Joined records: 2,511,464
+-------------------+--------------------+---------+-----------+-------------+
|    pickup_datetime|        LocationName|  Borough|fare_amount|trip_distance|
+-------------------+--------------------+---------+-----------+-------------+
|2025-08-01 00:52:23|   LaGuardia Airport|   Queens|       33.8|         8.44|
|2025-08-01 00:03:01|   LaGuardia Airport|   Queens|       21.2|         4.98|
|2025-08-01 00:25:34|        Central Park|Manhattan|       11.4|         2.14|
|2025-08-01 00:16:36|Greenwich Village...|Manhattan|       18.4|         3.06|
|2025-08-01 00:56:02|       Midtown North|Manhattan|       24.7|         5.25|
|2025-08-01 00:12:10|       Midtown North|Manhattan|        7.9|         1.45|
|2025-08-01 00:05:30|      Yorkville West|Manhattan|        5.8|         0.73|
|2025-08-01 00:11:29|        West Village|Manhattan|       12.1|         1.81|
|2025-08-01 00:41:02|West Chelsea/Huds...|Manhattan|       24.7|       

In [ ]:
from pyspark.sql.functions import broadcast

# Regular join (without broadcast)
start_time = time.time()
regular_join = df_enhanced.join(location_df,
                                 df_enhanced.PULocationID == location_df.LocationID)
regular_result = regular_join.count()
regular_time = time.time() - start_time

print(f"Regular join time: {regular_time:.2f} seconds")
print(f"Joined records: {regular_result:,}")

Regular join time: 0.89 seconds
Joined records: 2,511,464


In [ ]:
if regular_time > broadcast_time:
    speedup = (regular_time - broadcast_time) / regular_time * 100
    print(f"Performance improvement: {speedup:.1f}%")

In [ ]:
# Show sample of joined data with location names
broadcast_join.select('pickup_datetime', 'LocationName', 'Borough',
                      'fare_amount', 'trip_distance').show(10)

+-------------------+--------------------+---------+-----------+-------------+
|    pickup_datetime|        LocationName|  Borough|fare_amount|trip_distance|
+-------------------+--------------------+---------+-----------+-------------+
|2025-08-01 00:52:23|   LaGuardia Airport|   Queens|       33.8|         8.44|
|2025-08-01 00:03:01|   LaGuardia Airport|   Queens|       21.2|         4.98|
|2025-08-01 00:25:34|        Central Park|Manhattan|       11.4|         2.14|
|2025-08-01 00:16:36|Greenwich Village...|Manhattan|       18.4|         3.06|
|2025-08-01 00:56:02|       Midtown North|Manhattan|       24.7|         5.25|
|2025-08-01 00:12:10|       Midtown North|Manhattan|        7.9|         1.45|
|2025-08-01 00:05:30|      Yorkville West|Manhattan|        5.8|         0.73|
|2025-08-01 00:11:29|        West Village|Manhattan|       12.1|         1.81|
|2025-08-01 00:41:02|West Chelsea/Huds...|Manhattan|       24.7|         4.57|
|2025-08-01 00:49:28|   LaGuardia Airport|   Queens|

In [ ]:
print("=" * 60)
print("PERFORMANCE OPTIMIZATION SUMMARY")
print("=" * 60)
print(f"Caching improvement: {speedup:.1f}%")
print(f"Broadcast join improvement: [calculated above]")
print(f"Partitioning benefit: Faster filtering on pickup_hour")
print("=" * 60)

PERFORMANCE OPTIMIZATION SUMMARY
Caching improvement: 24.9%
Broadcast join improvement: [calculated above]
Partitioning benefit: Faster filtering on pickup_hour


## Statistical Analysis
Descriptive statistics, correlation matrix, and IQR-based outlier detection on fare amounts.

In [ ]:
df_enhanced.select('trip_distance', 'fare_amount', 'trip_duration_minutes',
                   'tip_amount', 'total_amount').describe().show()

+-------+------------------+------------------+---------------------+------------------+------------------+
|summary|     trip_distance|       fare_amount|trip_duration_minutes|        tip_amount|      total_amount|
+-------+------------------+------------------+---------------------+------------------+------------------+
|  count|           2511464|           2511464|              2511464|           2511464|           2511464|
|   mean|3.7276851230993837|20.495627729485598|    17.38201820531682|3.6371542773455796| 30.21432789799862|
| stddev| 5.426771629285761|19.564801192414688|    29.68010137950226| 4.203460991076329|24.152134268712494|
|    min|              0.01|              0.01|                 0.02|               0.0|              1.01|
|    max|           3900.78|             499.0|              6987.68|             222.0|            535.75|
+-------+------------------+------------------+---------------------+------------------+------------------+



In [ ]:
print(f"Trip Distance vs fare: {df_enhanced.stat.corr('trip_distance', 'fare_amount'):.3f}")
print(f"Trip Distance vs duration: {df_enhanced.stat.corr('trip_distance', 'trip_duration_minutes'):.3f}")
print(f"Fare vs tip: {df_enhanced.stat.corr('fare_amount', 'tip_amount'):.3f}")

Trip Distance vs fare: 0.827
Trip Distance vs duration: 0.361
Fare vs tip: 0.572


In [ ]:
quantiles = df_enhanced.approxQuantile('fare_amount', [0.25, 0.5, 0.75], 0.01)
Q1, median, Q3 = quantiles
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"Fare amount statistics:")
print(f"Q1 (25th percentile): ${Q1:.2f}")
print(f"Median (50th percentile): ${median:.2f}")
print(f"Q3 (75th percentile): ${Q3:.2f}")
print(f"IQR: ${IQR:.2f}")
print(f"Lower bound: ${lower_bound:.2f}")
print(f"Upper bound: ${upper_bound:.2f}")

outliers = df_enhanced.filter(
    (col('fare_amount') < lower_bound) | (col('fare_amount') > upper_bound)
)
print(f"Number of outliers: {outliers.count():,}")
print(f"Percentage of outliers: {(outliers.count() / df_enhanced.count() * 100):.2f}%")

Fare amount statistics:
Q1 (25th percentile): $9.30
Median (50th percentile): $13.50
Q3 (75th percentile): $22.60
IQR: $13.30
Lower bound: $-10.65
Upper bound: $42.55
Number of outliers: 277,729
Percentage of outliers: 11.06%


## Export for Visualization
Saving aggregated datasets (hourly, daily, locations, routes) as CSV files for Tableau.

In [ ]:
tableau_hourly = df_enhanced.groupBy('pickup_hour').agg(
    count('*').alias('trip_count'),
    round(avg('fare_amount'), 2).alias('avg_fare'),
    round(avg('trip_distance'), 2).alias('avg_distance'),
    round(avg('trip_duration_minutes'), 2).alias('avg_duration'),
    round(sum('total_amount'), 2).alias('total_revenue'),
    round(avg('tip_percentage'), 2).alias('avg_tip_pct')
).orderBy('pickup_hour')

tableau_hourly.show(24)

tableau_hourly.coalesce(1).write.mode('overwrite').csv('/content/tableau_hourly.csv', header=True)

+-----------+----------+--------+------------+------------+-------------+-----------+
|pickup_hour|trip_count|avg_fare|avg_distance|avg_duration|total_revenue|avg_tip_pct|
+-----------+----------+--------+------------+------------+-------------+-----------+
|          0|     68519|   22.07|        4.49|       15.19|   2213334.94|      20.69|
|          1|     45550|   20.11|        4.02|       13.98|   1350952.76|      20.52|
|          2|     29181|   17.89|        3.45|       12.71|    781758.55|      28.93|
|          3|     18789|   18.28|        3.52|       12.96|    512020.69|      20.22|
|          4|     12487|   23.87|        4.94|        15.8|    420056.97|      18.36|
|          5|     15478|   29.06|        6.58|       19.13|    605985.69|      16.35|
|          6|     33838|   23.85|        5.21|        17.6|   1096386.38|      17.55|
|          7|     60276|    21.0|        4.26|       17.26|   1779745.97|      21.03|
|          8|     86902|   19.33|        3.56|       1

In [ ]:
tableau_daily = df_enhanced.groupBy('pickup_day', 'day_name').agg(
    count('*').alias('trip_count'),
    round(avg('fare_amount'), 2).alias('avg_fare'),
    round(avg('trip_distance'), 2).alias('avg_distance'),
    round(avg('trip_duration_minutes'), 2).alias('avg_duration'),
    round(sum('total_amount'), 2).alias('total_revenue')
).orderBy('pickup_day')

tableau_daily.show()

tableau_daily.coalesce(1).write.mode('overwrite').csv('/content/tableau_daily.csv', header=True)

+----------+---------+----------+--------+------------+------------+-------------+
|pickup_day| day_name|trip_count|avg_fare|avg_distance|avg_duration|total_revenue|
+----------+---------+----------+--------+------------+------------+-------------+
|         1|   Sunday|    353013|   21.74|        4.18|        16.8|1.105949322E7|
|         2|   Monday|    295922|   21.16|        4.01|       17.39|    9235219.7|
|         3|  Tuesday|    338415|   19.79|        3.53|       17.17|1.000760259E7|
|         4|Wednesday|    371998|   19.65|        3.38|       17.77|1.095804063E7|
|         5| Thursday|    351619|   20.65|        3.66|       18.21| 1.07679829E7|
|         6|   Friday|    407527|   20.72|        3.77|        17.9|1.249038068E7|
|         7| Saturday|    392970|   19.92|        3.63|       16.44|1.136347708E7|
+----------+---------+----------+--------+------------+------------+-------------+



In [ ]:
tableau_locations = df_enhanced.groupBy('PULocationID').agg(
    count('*').alias('pickup_count'),
    round(avg('fare_amount'), 2).alias('avg_fare'),
    round(avg('trip_distance'), 2).alias('avg_distance'),
    round(sum('total_amount'), 2).alias('total_revenue')
).orderBy(col('pickup_count').desc())

tableau_locations.show(20)

tableau_locations.coalesce(1).write.mode('overwrite').csv('/content/tableau_locations.csv', header=True)

+------------+------------+--------+------------+-------------+
|PULocationID|pickup_count|avg_fare|avg_distance|total_revenue|
+------------+------------+--------+------------+-------------+
|         132|      169442|   62.06|       15.07|1.342458164E7|
|         161|      120420|   15.92|        2.31|    3038956.5|
|         237|      109051|   12.68|        1.81|   2251722.15|
|         186|      103192|   16.72|         2.4|   2646025.98|
|         138|       93466|   44.24|        9.52|   6373629.21|
|         162|       92670|    15.2|        2.27|   2249264.27|
|         230|       86566|   18.92|        3.05|   2491446.91|
|         236|       84462|   13.28|        2.04|   1785300.95|
|         170|       77805|   15.22|        2.27|   1875622.82|
|         163|       69195|   16.07|        2.42|   1752116.09|
|          68|       69147|    16.9|        2.54|   1804219.63|
|         234|       68764|   14.01|        2.03|   1565797.56|
|         142|       65393|   13.74|    

In [ ]:
tableau_routes = df_enhanced.groupBy('PULocationID', 'DOLocationID').agg(
    count('*').alias('route_count'),
    round(avg('fare_amount'), 2).alias('avg_fare'),
    round(avg('trip_distance'), 2).alias('avg_distance')
).filter(col('route_count') > 50).orderBy(col('route_count').desc())

tableau_routes.show(20)

tableau_routes.coalesce(1).write.mode('overwrite').csv('/content/tableau_routes.csv', header=True)

+------------+------------+-----------+--------+------------+
|PULocationID|DOLocationID|route_count|avg_fare|avg_distance|
+------------+------------+-----------+--------+------------+
|         237|         236|      13648|    8.18|        1.04|
|         236|         237|      10848|    8.67|        1.02|
|         237|         237|      10120|    7.06|        0.63|
|         132|         265|       9486|  103.88|       20.39|
|         161|         237|       8211|     9.5|        1.06|
|         236|         236|       8014|    6.48|        0.61|
|         237|         161|       7612|    9.75|        1.05|
|         132|         230|       6500|   71.01|       17.76|
|         161|         236|       6260|    13.0|        1.89|
|         186|         230|       6251|   12.63|        1.06|
|         237|         162|       6137|    9.13|        0.99|
|         142|         239|       5500|    7.77|        0.96|
|         239|         142|       5388|    7.44|        0.87|
|       